# Conversational agent

In [ ]:
# 读取 .env 里的 OPENAI_API_KEY
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [ ]:
# @tool 装饰器：把普通函数包装成 LangChain 的 Tool 对象。
# 注：langchain.tools 这个入口目前仍然可用（它转发自 langchain_core.tools），不用改
from langchain.tools import tool

In [ ]:
# 和 L5 一样：查询实时天气的工具，用 Open-Meteo 免费公开 API（不需要 key）
import requests
import datetime
from pydantic import BaseModel,Field
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""

    BASE_URL = "https://api.open-meteo.com/v1/forecast"

    # Parameters for the request
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }
     # Make the request
    response=requests.get(BASE_URL,params=params)
    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")
    current_utc_time=datetime.datetime.utcnow()
    time_list = [datetime.datetime.fromisoformat(time_str.replace('Z', '+00:00')) for time_str in results['hourly']['time']]
    temperature_list = results['hourly']['temperature_2m']

    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]

    return f'The current temperature is {current_temperature}°C'

In [ ]:
import wikipedia

# TODO: 请在此处补全代码（同 L5）
# 用 @tool 装饰 search_wikipedia(query: str) -> str，注意异常要用 wikipedia.exceptions.PageError /
# wikipedia.exceptions.DisambiguationError（不要写成 self.wiki_client.xxx）
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries."""
    # TODO: 请在此处补全代码
    pass

In [ ]:
# 汇总所有可用工具，后面统一转换成 OpenAI function schema 并绑定给模型
tools = [get_current_temperature, search_wikipedia]

In [ ]:
# 【版本兼容修复】以下几个都因为 langchain 拆包而搬了家：
#   ChatOpenAI                        -> langchain_openai
#   ChatPromptTemplate                -> langchain_core.prompts
#   format_tool_to_openai_function     -> langchain_community.tools.render
#   OpenAIFunctionsAgentOutputParser  -> langchain_classic.agents.output_parsers（旧版 agent 相关代码归到了 langchain_classic）
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.render import format_tool_to_openai_function
from langchain_classic.agents.output_parsers import OpenAIFunctionsAgentOutputParser

In [ ]:
# TODO: 请在此处补全代码
# 组装一条最基础的 function-calling chain：
#   functions = [format_tool_to_openai_function(f) for f in tools]
#   model = ChatOpenAI(temperature=0).bind(functions=functions)
#   prompt = ChatPromptTemplate.from_messages([("system", "You are helpful but sassy assistant"), ("user", "{input}")])
#   chain = prompt | model | OpenAIFunctionsAgentOutputParser()
functions = None  # TODO: 请在此处补全代码
model = None  # TODO: 请在此处补全代码
prompt = None  # TODO: 请在此处补全代码
chain = None  # TODO: 请在此处补全代码

In [ ]:
result = chain.invoke({"input": "what is the weather is sf?"})

In [ ]:
result.tool

In [ ]:
result.tool_input

In [ ]:
# 【版本兼容修复】langchain.prompts 已不存在，MessagesPlaceholder 搬到了 langchain_core.prompts
# MessagesPlaceholder(variable_name="agent_scratchpad")：给"中间步骤"（之前调用过哪些工具、结果是什么）
# 预留一个位置，模型能看到自己之前做过的操作，从而支持多轮工具调用
from langchain_core.prompts import MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [ ]:
chain = prompt | model | OpenAIFunctionsAgentOutputParser()

In [ ]:
# 第一轮：还没有任何中间步骤，所以 agent_scratchpad 传空列表
result1 = chain.invoke({
    "input": "what is the weather is sf?",
    "agent_scratchpad": []
})

In [ ]:
result1.tool

In [ ]:
# 提示：新版 langchain_core 里 StructuredTool 不支持直接 __call__，
# 直接 get_current_temperature(result1.tool_input) 会报错，要用 .invoke(...)
# TODO: 请在此处补全代码
observation = None  # TODO: 请在此处补全代码

In [ ]:
observation

In [ ]:
# result1 的类型是 AgentActionMessageLog，message_log 里保留了原始的模型消息（用于拼回 scratchpad）
type(result1)

In [ ]:
# 【版本兼容修复】langchain.agents.format_scratchpad 已不存在，搬到了 langchain_classic.agents.format_scratchpad
# format_to_openai_functions：把 (AgentAction, observation) 这样的元组列表，
# 转换成 OpenAI function-calling 消息格式（assistant 的 function_call + function 角色的结果），
# 填进 MessagesPlaceholder(variable_name="agent_scratchpad") 里
from langchain_classic.agents.format_scratchpad import format_to_openai_functions

In [ ]:
result1.message_log

In [ ]:
# 看看 (result1, observation) 这一条中间步骤被格式化成什么样的消息列表
format_to_openai_functions([(result1, observation), ])

In [ ]:
# 第二轮：把第一轮的 (result1, observation) 转成 scratchpad 传进去，
# 这样模型能"看到"自己已经查过天气、结果是什么，从而给出最终回答（而不是再调用一次工具）
result2 = chain.invoke({
    "input": "what is the weather is sf?",
    "agent_scratchpad": format_to_openai_functions([(result1, observation)])
})

In [ ]:
# 期望：result2 这次是 AgentFinish（已经拿到天气结果了，不需要再调用工具）
result2

In [ ]:
from langchain_core.agents import AgentFinish

# TODO: 请在此处补全代码
# 手写一个简易 Agent 循环 run_agent(user_input)：
#   intermediate_steps = []
#   while True:
#       result = chain.invoke({"input": user_input, "agent_scratchpad": format_to_openai_functions(intermediate_steps)})
#       如果 isinstance(result, AgentFinish)：return result
#       否则：根据 result.tool 从 {"search_wikipedia": search_wikipedia, "get_current_temperature": get_current_temperature}
#             找到工具，用 .run(result.tool_input) 执行，得到 observation
#       intermediate_steps.append((result, observation))
def run_agent(user_input):
    # TODO: 请在此处补全代码
    pass

In [ ]:
# TODO: 请在此处补全代码
# RunnablePassthrough.assign(agent_scratchpad=lambda x: format_to_openai_functions(x["intermediate_steps"])) | chain
from langchain_core.runnables import RunnablePassthrough
agent_chain = None  # TODO: 请在此处补全代码

In [ ]:
# TODO: 请在此处补全代码
# 用 agent_chain 重写 run_agent：调用方只需传 {"input": user_input, "intermediate_steps": intermediate_steps}
def run_agent(user_input):
    # TODO: 请在此处补全代码
    pass

In [ ]:
run_agent("what is the weather is sf?")

In [ ]:
run_agent("what is langchain?")

In [ ]:
run_agent("hi!")

In [ ]:
# TODO: 请在此处补全代码
# from langchain_classic.agents import AgentExecutor
# agent_executor = AgentExecutor(agent=agent_chain, tools=tools, verbose=True)
agent_executor = None  # TODO: 请在此处补全代码

In [ ]:
agent_executor.invoke({"input": "what is langchain?"})

In [ ]:
agent_executor.invoke({"input": "my name is bob"})

In [ ]:
# 注意：这个 agent_executor 目前还没有接"记忆"（memory），
# 每次 invoke 都是独立的一轮对话，所以模型不会记得上一句说的名字——下面几个 cell 会加上 memory 来解决这个问题
agent_executor.invoke({"input": "what is my name"})

In [ ]:
# 在 prompt 里加一个 MessagesPlaceholder(variable_name="chat_history")，
# 用来存放"之前几轮对话的历史消息"（不同于 agent_scratchpad 存的是"当前这一轮内部的工具调用记录"）
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [ ]:
# TODO: 请在此处补全代码：用新 prompt（带 chat_history）重新组装 agent_chain
agent_chain = None  # TODO: 请在此处补全代码

In [ ]:
# TODO: 请在此处补全代码
# from langchain_classic.memory import ConversationBufferMemory
# memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")
memory = None  # TODO: 请在此处补全代码

In [ ]:
# 把 memory 传给 AgentExecutor，这样每轮对话都会自动读写 chat_history，不需要我们手动维护
agent_executor = AgentExecutor(agent=agent_chain, tools=tools, verbose=True, memory=memory)

In [ ]:
agent_executor.invoke({"input": "my name is bob"})

In [ ]:
# 期望：这次模型能记得上一轮说过 "my name is bob"，回答 bob
agent_executor.invoke({"input": "whats my name"})

In [ ]:
agent_executor.invoke({"input": "whats the weather in sf?"})

In [ ]:
# 一个占位用的自定义工具示例：把输入字符串反转返回，方便你替换成自己想要的逻辑
@tool
def create_your_own(query: str) -> str:
    """This function can do whatever you would like once you fill it in """
    print(type(query))
    return query[::-1]

In [ ]:
tools = [get_current_temperature, search_wikipedia, create_your_own]

In [ ]:
# 【环境限制，未擅自安装新依赖】panel 是一个第三方 GUI 库，用来在 Jupyter 里快速搭一个聊天窗口。
# 当前 venv 没有安装 panel（不在题目给出的已装包清单里），import 会直接 ModuleNotFoundError。
# 如果想跑通这一节，需要额外执行 `pip install panel`（未经你确认，这里没有擅自安装）。
# 下面的代码逻辑保持课程原本写法不变，只是在当前环境下 import 这一步就会失败。
import panel as pn  # GUI
pn.extension()
import panel as pn
import param

# cbfs：把"构建 agent_executor" + "维护对话历史" + "渲染成聊天气泡"都封装进一个类里，
# 方便和下面的 Panel 交互控件绑定
class cbfs(param.Parameterized):

    def __init__(self, tools, **params):
        super(cbfs, self).__init__( **params)
        self.panels = []
        self.functions = [format_tool_to_openai_function(f) for f in tools]
        self.model = ChatOpenAI(temperature=0).bind(functions=self.functions)
        self.memory = ConversationBufferMemory(return_messages=True,memory_key="chat_history")
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are helpful but sassy assistant"),
            MessagesPlaceholder(variable_name="chat_history"),
            ("user", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])
        self.chain = RunnablePassthrough.assign(
            agent_scratchpad = lambda x: format_to_openai_functions(x["intermediate_steps"])
        ) | self.prompt | self.model | OpenAIFunctionsAgentOutputParser()
        self.qa = AgentExecutor(agent=self.chain, tools=tools, verbose=False, memory=self.memory)

    def convchain(self, query):
        # convchain：Panel 文本框每次提交内容时会调用这个方法，
        # 执行一次 agent，并把"用户消息 + ChatBot 回复"追加成两行气泡渲染出来
        if not query:
            return
        inp.value = ''
        result = self.qa.invoke({"input": query})
        self.answer = result['output']
        self.panels.extend([
            pn.Row('User:', pn.pane.Markdown(query, width=450)),
            pn.Row('ChatBot:', pn.pane.Markdown(self.answer, width=450, styles={'background-color': '#F6F6F6'}))
        ])
        return pn.WidgetBox(*self.panels, scroll=True)


    def clr_history(self,count=0):
        self.chat_history = []
        return

In [ ]:
# 组装出实际的 Panel 页面：一个输入框 + 一个滚动显示对话记录的区域
cb = cbfs(tools)

inp = pn.widgets.TextInput( placeholder='Enter text here…')

conversation = pn.bind(cb.convchain, inp)

tab1 = pn.Column(
    pn.Row(inp),
    pn.layout.Divider(),
    pn.panel(conversation,  loading_indicator=True, height=400),
    pn.layout.Divider(),
)

dashboard = pn.Column(
    pn.Row(pn.pane.Markdown('# QnA_Bot')),
    pn.Tabs(('Conversation', tab1))
)
dashboard